In [44]:
import matplotlib.pyplot as plt
import numpy as np
import os
from pathlib import Path
from dotenv import load_dotenv
from spotify_api import get_access_token as get_spotify_access_token
from tidal_api import create_session as create_tidal_session

load_dotenv()

SPOTIFY_CLIENT_ID = os.getenv('SPOTIFY_CLIENT_ID')
SPOTIFY_CLIENT_SECRET = os.getenv('SPOTIFY_CLIENT_SECRET')
if not SPOTIFY_CLIENT_ID or not SPOTIFY_CLIENT_SECRET:
    raise ValueError("Spotify client ID and secret must be set in environment variables.")

SPOT_HEADERS = get_spotify_access_token(SPOTIFY_CLIENT_ID, SPOTIFY_CLIENT_SECRET)
print("Authenticated with Spotify successfully.")

TIDAL_SESSION = create_tidal_session(Path("tidal_session.json"))
print("Authenticated with TIDAL successfully.")

Authenticated with Spotify successfully.
Authenticated with TIDAL successfully.


In [5]:
import pandas as pd

df = pd.read_csv('Songs.csv')
df.head()

,RowID,artist_names,artists_id,danceability,energy,valence,tempo,loudness,mode,key,...,instrumentalness,liveness,speechiness,explicit,duration_ms,popularity,year,release_date,song_title (censored),Simpler_Label
0,1,['Beyonce'],[761179],0.476,0.688,0.352,140.553,-7.799,1,1,...,0.000645,0.4380,0.2450,1,250960,62,2014,11/24/14,***Flawless (feat. Chimamanda Ngozi Adichie),B-1
1,2,['Beyonce'],[761179],0.574,0.573,0.626,136.304,-7.079,1,11,...,0.000022,0.0933,0.0850,0,322000,67,2016,4/23/16,All Night,B-2
2,3,['Beyonce'],[761179],0.713,0.842,0.435,102.968,-5.424,0,6,...,0.000000,0.3410,0.0984,0,240777,62,2019,4/17/19,Before I Let Go - Homecoming Live Bonus Track,B-3
3,4,['Beyonce'],[761179],0.545,0.649,0.297,99.099,-4.062,1,6,...,0.000016,0.0894,0.0324,0,253747,70,2011,6/24/11,Best Thing I Never Had,B-4
4,5,['Beyonce'],[761179],0.665,0.787,0.699,167.349,-5.730,1,5,...,0.000000,0.2190,0.1510,0,212147,66,2011,6/24/11,Countdown,B-5


In [6]:
from spotify_api import get_track_id
import ast

track_ids = []
for _, row in df.iterrows():
    artists = ast.literal_eval(row['artist_names'])
    artist = artists[0] if artists else ""
    song = row['song_title (censored)']
    tid = get_track_id(artist, song, SPOT_HEADERS)
    track_ids.append(tid)

print(f"Found {sum(1 for t in track_ids if t):,} / {len(track_ids):,} track IDs")
track_ids[:10]

Found 100 / 100 track IDs


['7tefUew2RUuSAqHyegMoY1',
 '7oAuqs6akGnPU3Tb00ZmyM',
 '7LikBkHerFGZ58QHVOKp1t',
 '3lBRNqXjPp2j3JMTCXDTNO',
 '3axkNosdVQLZiq1HakuGhc',
 '2f4IuijXLxYOeBncS60GUD',
 '71OvX5NNLrmz7rpq1ANTQn',
 '7cvkXf3AwPGT041PyOi5VX',
 '5fyIGoaaKelzdyW8ELhYJZ',
 '0zVMzJ37VQNFUNvdxxat2E']

In [8]:
from tidal_api import spotify_to_tidal_ids

tidal_song_ids: list[dict] = spotify_to_tidal_ids(track_ids, SPOT_HEADERS, TIDAL_SESSION)
print(tidal_song_ids[:10])

Searching TIDAL by name (fallback): 100%|██████████| 3/3 [00:00<00:00, 15.05it/s]


[{'spotify_id': '7tefUew2RUuSAqHyegMoY1', 'isrc': 'USSM11307808', 'tidal_id': '37936041'}, {'spotify_id': '7oAuqs6akGnPU3Tb00ZmyM', 'isrc': 'USSM11603185', 'tidal_id': '59727867'}, {'spotify_id': '7LikBkHerFGZ58QHVOKp1t', 'isrc': 'USSM11902768', 'tidal_id': '107780125'}, {'spotify_id': '3lBRNqXjPp2j3JMTCXDTNO', 'isrc': 'USSM11102904', 'tidal_id': '11624863'}, {'spotify_id': '3axkNosdVQLZiq1HakuGhc', 'isrc': 'USSM11102909', 'tidal_id': '19646524'}, {'spotify_id': '2f4IuijXLxYOeBncS60GUD', 'isrc': 'USSM11410272', 'tidal_id': '50805406'}, {'spotify_id': '71OvX5NNLrmz7rpq1ANTQn', 'isrc': 'USSM11603180', 'tidal_id': '108043428'}, {'spotify_id': '7cvkXf3AwPGT041PyOi5VX', 'isrc': 'USSM11102795', 'tidal_id': '19646526'}, {'spotify_id': '5fyIGoaaKelzdyW8ELhYJZ', 'isrc': 'USCM51400377', 'tidal_id': '79414000'}, {'spotify_id': '0zVMzJ37VQNFUNvdxxat2E', 'isrc': 'USSM11406084', 'tidal_id': '37936046'}]


# Why am I using a database?
I'm using Postgres because it's a simple and fast and holds to a schema better than a CSV file. This allows me to add the additional features across multiple Python sessions without having to store all the data in memory or constantly exporting it to a CSV file.

In [9]:
from db_connection import db, Songs, SongAudioFeatures

db.connect(reuse_if_open=True)
db.create_tables([Songs, SongAudioFeatures], safe=True)
print("Tables ready.")
db.close()

Tables ready.


True

In [52]:
import uuid
from spotify_api import get_tracks_metadata

tracks_meta = get_tracks_metadata(track_ids, SPOT_HEADERS)
print(f"Fetched metadata for {len(tracks_meta)} tracks")

tidal_lookup = {entry["spotify_id"]: entry["tidal_id"] for entry in tidal_song_ids}

db.connect(reuse_if_open=True)
inserted, skipped = 0, 0
for meta in tracks_meta:
    try:
        Songs.create(
            id=uuid.uuid4(),
            spotify_id=meta["id"],
            tidal_id=tidal_lookup.get(meta["id"]),
            song_name=meta["name"],
            artists=meta["artists"],
            album_name=meta.get("album_name"),
            album_total_tracks=meta.get("album_total_tracks"),
            release_date=meta.get("album_release_date"),
            disc_number=meta.get("disc_number"),
            duration_ms=meta.get("duration_ms"),
            explicit=meta.get("explicit"),
            isrc=meta.get("isrc"),
            spotify_external_url=meta.get("external_url"),
            spotify_href=meta.get("href"),
            popularity=meta.get("popularity"),
            track_number=meta.get("track_number"),
            spotify_uri=meta.get("uri"),
        )
        inserted += 1
    except Exception as e:
        skipped += 1

db.close()

Fetched metadata for 100 tracks


True

# Why am I retrieving all the songs?
Although Spotify's API is great for getting audio features, it's quite limited and doesn't include things like lyrics and audio features that are hard to compute. With the raw audio data, we can compute these features ourselves, while opening up the possibility to generate lyrics transcriptions with language models like Whisper for tracks that don't have them.

In [15]:
from tidal_api import download_songs

downloaded_paths = download_songs(tidal_song_ids)
print(f"Downloaded {len(downloaded_paths)} tracks")

Downloaded 91 tracks


In [21]:
for i in [i for i in tidal_song_ids if i.get("tidal_id") not in [x.split("/")[-1].split(".")[0] for x in downloaded_paths]]:
    if os.path.exists(f"OrpheusDL/source2/{i['spotify_id']}.m4a"):
        os.rename(f"OrpheusDL/source2/{i['spotify_id']}.m4a", f"OrpheusDL/downloads/{i['tidal_id']}.m4a")
        downloaded_paths.append(f"OrpheusDL/downloads/{i['tidal_id']}.m4a")
    else:
        print(f"File not found: {i['spotify_id']}.m4a")

In [35]:
from audio_processor import process_bulk

analysis_results = process_bulk([i for i in downloaded_paths if os.path.exists(i)])

Processing tracks: 100%|██████████| 106/106 [01:22<00:00,  1.29track/s]


In [55]:
existing_paths = [i for i in downloaded_paths if os.path.exists(i)]
tidal_ids_list = [p.split("/")[-1].split(".")[0] for p in existing_paths]

db.connect(reuse_if_open=True)
stored, failed = 0, 0
for tid, analysis in zip(tidal_ids_list, analysis_results):
    if "error" in analysis:
        print(f"[SKIP] {tid}: {analysis['error']}")
        failed += 1
        continue

    song = Songs.get_or_none(Songs.tidal_id == tid)
    if song is None:
        failed += 1
        continue

    track_info = analysis.get("track", {})
    try:
        SongAudioFeatures.create(
            song=song.id,
            acousticness=analysis.get("acousticness"),
            danceability=analysis.get("danceability"),
            energy=analysis.get("energy"),
            instrumentalness=analysis.get("instrumentalness"),
            key=analysis.get("key"),
            liveness=analysis.get("liveness"),
            loudness=analysis.get("loudness"),
            mode=analysis.get("mode"),
            speechiness=analysis.get("speechiness"),
            tempo=analysis.get("tempo"),
            time_signature=analysis.get("time_signature"),
            valence=analysis.get("valence"),
            num_samples=track_info.get("num_samples"),
            duration=track_info.get("duration"),
            sample_md5=track_info.get("sample_md5"),
            end_of_fade_in=track_info.get("end_of_fade_in"),
            start_of_fade_out=track_info.get("start_of_fade_out"),
            tempo_confidence=track_info.get("tempo_confidence"),
            time_signature_confidence=track_info.get("time_signature_confidence"),
            key_confidence=track_info.get("key_confidence"),
            mode_confidence=track_info.get("mode_confidence"),
            bars=analysis.get("bars"),
            beats=analysis.get("beats"),
            sections=analysis.get("sections"),
            segments=analysis.get("segments"),
            tatums=analysis.get("tatums"),
        )
        stored += 1
    except Exception as e:
        failed += 1

db.close()

True

In [56]:
from tidal_api import get_track_lyrics
from tqdm.auto import tqdm

db.connect(reuse_if_open=True)
songs_needing_lyrics = (
    Songs.select()
    .where(Songs.tidal_id.is_null(False) & (Songs.lyrics.is_null(True) | (Songs.lyrics == "")))
)
songs_list = list(songs_needing_lyrics)
print(f"Songs needing lyrics: {len(songs_list)}")

fetched, missed = 0, 0
for song in tqdm(songs_list, desc="Fetching lyrics"):
    lyrics = get_track_lyrics(song.tidal_id, TIDAL_SESSION)
    if lyrics:
        Songs.update(lyrics=lyrics).where(Songs.id == song.id).execute()
        fetched += 1
    else:
        missed += 1

db.close()
print(f"Fetched lyrics for {fetched} songs, {missed} not available")

Songs needing lyrics: 100


Fetching lyrics: 100%|██████████| 100/100 [00:16<00:00,  6.06it/s]

Fetched lyrics for 66 songs, 34 not available


In [57]:
from large_language_model import transcribe_songs
from db_connection import db, Songs

db.connect(reuse_if_open=True)
still_missing = (
    Songs.select()
    .where(Songs.tidal_id.is_null(False) & (Songs.lyrics.is_null(True) | (Songs.lyrics == "")))
)
songs_to_transcribe = [
    {"id": str(s.id), "tidal_id": s.tidal_id}
    for s in still_missing
]
transcribe_songs(songs_to_transcribe)
db.close()

Transcribing songs: 100%|██████████| 34/34 [00:41<00:00,  1.21s/it]


In [58]:
from embedding_model import create_text_embeddings

db.connect(reuse_if_open=True)
songs_with_lyrics = (
    Songs.select(Songs, SongAudioFeatures)
    .join(SongAudioFeatures, on=(SongAudioFeatures.song == Songs.id))
    .where(
        Songs.lyrics.is_null(False)
        & (Songs.lyrics != "")
        & SongAudioFeatures.lyrics_embeddings.is_null(True)
    )
)
songs_list = list(songs_with_lyrics)

embedded = 0
for song in tqdm(songs_list, desc="Generating Lyrics embeddings"):
    lines = [line.strip() for line in song.lyrics.splitlines() if line.strip()]
    if not lines:
        continue
    line_embeddings = create_text_embeddings(lines)           # (n_lines, 768)
    song_embedding = line_embeddings.mean(dim=0)              # (768,)
    song_embedding = song_embedding / song_embedding.norm()   # normalize embeddings
    (SongAudioFeatures
     .update(lyrics_embeddings=song_embedding.cpu().tolist())
     .where(SongAudioFeatures.song == song.id)
     .execute())
    embedded += 1

db.close()

Generating Lyrics embeddings: 100%|██████████| 100/100 [02:05<00:00,  1.25s/it]

Stored lyrics embeddings for 100 songs


In [59]:
from file_metadata_processor import extract_image_from_m4a_mutagen
from embedding_model import create_images_embeddings

db.connect(reuse_if_open=True)
songs_needing_images = (
    Songs.select(Songs, SongAudioFeatures)
    .join(SongAudioFeatures, on=(SongAudioFeatures.song == Songs.id))
    .where(
        Songs.tidal_id.is_null(False)
        & SongAudioFeatures.image_embeddings.is_null(True)
    )
)

songs_list = list(songs_needing_images)
embedded = 0
batch_songs = []
batch_images = []

def process_batch():
    global embedded
    if not batch_images:
        return
    embeddings = create_images_embeddings(batch_images).cpu().tolist()
    for s, emb in zip(batch_songs, embeddings):
        SongAudioFeatures.update(image_embeddings=emb).where(SongAudioFeatures.song == s.id).execute()
        embedded += 1
    batch_songs.clear()
    batch_images.clear()

for song in tqdm(songs_list, desc="Generating image embeddings"):
    audio_path = downloads_dir / f"{song.tidal_id}.m4a"
    if not audio_path.exists():
        continue
    img = extract_image_from_m4a_mutagen(str(audio_path))
    if img is None:
        continue
    batch_songs.append(song)
    batch_images.append(img)
    if len(batch_images) >= 32:
        process_batch()

process_batch()

db.close()

Generating image embeddings: 100%|██████████| 100/100 [00:17<00:00,  5.58it/s]


True

In [61]:
import pandas as pd
from db_connection import db, Songs, SongAudioFeatures

tidal_ids = [entry["tidal_id"] for entry in tidal_song_ids]

db.connect(reuse_if_open=True)

songs_df = pd.DataFrame(list(Songs.select().where(Songs.tidal_id.in_(tidal_ids)).dicts()))
audio_features_df = pd.DataFrame(list(SongAudioFeatures.select().dicts()))

db.close()

merged_df = songs_df.merge(audio_features_df, left_on="id", right_on="song", how="left")
merged_df.drop(columns=["song"], inplace=True)

merged_df.to_csv("songs_export.csv", index=False)
merged_df.head()

,id_x,spotify_id,tidal_id,song_name,artists,album_name,album_total_tracks,release_date,disc_number,duration_ms_x,...,time_signature_confidence,key_confidence,mode_confidence,bars,beats,sections,segments,tatums,image_embeddings,lyrics_embeddings
0,38fcebd3-1368-4afb-afe2-b7edaf8c29b0,1fzAuUVbzlhZ1lJAx9PtY6,115983362,Daylight,[Taylor Swift],Lover,18,2019-08-23,1,293453,...,0.030,0.991,0.892,"[{'start': 0.25542, 'duration': 3.76163, 'conf...","[{'start': 0.25542, 'duration': 0.4644, 'confi...","[{'start': 0.0, 'duration': 4.92263, 'confiden...","[{'start': 0.0, 'duration': 0.20898, 'confiden...","[{'start': 0.25542, 'duration': 0.2322, 'confi...","[0.025611604, 0.005921087, 0.013295292, -0.032...","[0.010736673, -0.008948794, 0.0018226283, 0.00..."
1,020d124a-eeb4-4c3a-8f6b-9f1f7dddeb5c,6oVxXO5oQ4pTpO8RSnkzvv,100366423,Dress,[Taylor Swift],reputation,15,2017-11-10,1,230373,...,0.016,0.991,0.892,"[{'start': 2.36844, 'duration': 2.50776, 'conf...","[{'start': 2.36844, 'duration': 0.53406, 'conf...","[{'start': 0.0, 'duration': 20.06204, 'confide...","[{'start': 0.0, 'duration': 4.29569, 'confiden...","[{'start': 2.36844, 'duration': 0.26703, 'conf...","[0.010997367, -0.0068842694, -0.0025671981, 0....","[0.0075813867, -0.0025842197, -0.0010379436, -..."
2,1a0ef1b8-5fe7-41dd-8ccf-9e24ba3e9fb7,7HuBDWi18s4aJM8UFnNheH,100366419,King Of My Heart,[Taylor Swift],reputation,15,2017-11-10,1,214320,...,0.011,0.990,0.891,"[{'start': 0.20898, 'duration': 3.41333, 'conf...","[{'start': 0.20898, 'duration': 0.67338, 'conf...","[{'start': 0.0, 'duration': 6.68735, 'confiden...","[{'start': 0.0, 'duration': 0.16254, 'confiden...","[{'start': 0.20898, 'duration': 0.33669, 'conf...","[0.026619554, 0.0039208652, 0.013463227, -0.03...","[-0.0014199425, -0.0014469402, 0.0057828887, 0..."
3,a507334a-6a2a-4a4a-8138-1ecfc3f9b205,7tefUew2RUuSAqHyegMoY1,37936041,***Flawless (feat. Chimamanda Ngozi Adichie),"[Beyoncé, Chimamanda Ngozi Adichie]",BEYONCÉ [Platinum Edition],20,2014-11-24,1,250960,...,0.013,0.990,0.891,"[{'start': 0.30186, 'duration': 4.01705, 'conf...","[{'start': 0.30186, 'duration': 0.60372, 'conf...","[{'start': 0.0, 'duration': 9.3112, 'confidenc...","[{'start': 0.0, 'duration': 0.27864, 'confiden...","[{'start': 0.30186, 'duration': 0.30186, 'conf...","[-0.004235378, 0.008314926, -0.03128719, -0.02...","[0.010941954, 0.0028168878, 0.0012912183, 0.00..."
4,f7f0aa07-995a-4d8d-897b-70bfeef59109,7oAuqs6akGnPU3Tb00ZmyM,59727867,All Night,[Beyoncé],Lemonade,13,2016-04-23,1,322000,...,0.031,0.980,0.882,"[{'start': 0.2322, 'duration': 2.57741, 'confi...","[{'start': 0.2322, 'duration': 0.83592, 'confi...","[{'start': 0.0, 'duration': 6.10685, 'confiden...","[{'start': 0.0, 'duration': 1.0449, 'confidenc...","[{'start': 0.2322, 'duration': 0.41796, 'confi...","[0.003584969, -0.026952168, -0.015451225, -0.0...","[0.006416559, -0.0012309648, 0.0026711188, 0.0..."
